# 🧬 Docker

**Python, Datos e Ingeniería de IA Aplicada**  
UTN Rosario — FRRO, Área de Extensión Universitaria — 2026

**Encuentro 3 · Bloque 2 — 60 minutos**

Docente: **Matías Barreto** — [matiasbarreto.com](https://matiasbarreto.com)

---

## Objetivo

Entender qué es Docker, por qué resuelve el problema del "en mi máquina funciona", y cómo usarlo para empaquetar y distribuir aplicaciones Python de forma reproducible.

## Al terminar este bloque van a poder:

1. Distinguir imagen, contenedor, Hub y Dockerfile.
2. Usar los comandos básicos de Docker desde la terminal.
3. Pasar variables de entorno a un contenedor con `--env-file`.
4. Escribir un `Dockerfile` mínimo para empaquetar una app Python.
5. Servir una app Gradio con Gemini API desde dentro de un contenedor.

---

> **◈ Capítulo 3 de la cursada**
> En el bloque anterior vimos la diferencia entre entorno local y contenedor. Ahora lo hacemos: instalamos Docker, ejecutamos los comandos esenciales y empaquetamos una app Python real.

---

## 1. El problema que Docker resuelve

Imaginemos esta situación: terminamos de armar un script que consume la Gemini API y funciona perfecto en nuestra computadora. Se lo mandamos a un compañero. Él lo ejecuta y explota con un error de versión de Python, o de una librería que tiene diferente.

El problema no es el código. El problema es que **cada entorno local es diferente**.

### La analogía del contenedor de barco

Antes de los contenedores de carga estandarizados, mover mercancía entre países era un caos. Cada producto llegaba en su propio embalaje, en su propio formato. Los estibadores tenían que reorganizar todo en cada puerto.

Con los contenedores estandarizados, todo cambió: se carga una vez y viaja igual por barco, tren o camión. El puerto de destino lo recibe y lo maneja sin sorpresas.

**Docker hace exactamente lo mismo con el software.** Empaquetamos la aplicación —código, dependencias, versión de Python, configuración— dentro de un contenedor. Ese contenedor corre igual en cualquier máquina que tenga Docker instalado.

| Problema sin Docker | Solución con Docker |
|---|---|
| "En mi máquina funciona" | El contenedor lleva su propio entorno |
| Instalar 10 dependencias a mano | Un solo `docker run` |
| Conflictos de versiones de Python | Cada contenedor tiene la suya |
| Reproducir el entorno de producción | El `Dockerfile` es la receta exacta |

---

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Imagen** | La plantilla inmutable que define el contenedor: SO base, librerías, código. Una vez creada, no cambia. |
| **Contenedor** | Una instancia en ejecución de una imagen. Podemos tener varios contenedores corriendo desde la misma imagen. |
| **Docker Hub** | El repositorio público de imágenes. Como PyPI, pero para contenedores. |
| **Dockerfile** | Archivo de texto que describe cómo construir una imagen. La "receta": qué imagen base usar, qué instalar, qué copiar. |
| **Variable de entorno** | Valor de configuración que se pasa al contenedor desde afuera, sin escribirlo en el código. Las API keys viajan así. |

### ✎ Para pensar antes de avanzar

¿Qué diferencia hay entre una imagen y un contenedor? ¿Se parece a alguna diferencia que ya conocen en Python? Anoten su respuesta antes de leer la explicación de la siguiente sección.

---

## 3. Imagen vs. contenedor — la analogía de la receta

Una **imagen** es como una receta de cocina: es el plano, la descripción de qué contiene y cómo se construye. La receta no cocina sola.

Un **contenedor** es el plato ya preparado, corriendo en la mesa. Podemos preparar diez platos a partir de la misma receta, todos iguales.

En Python ya vimos algo parecido:

- La **clase** es la receta (imagen).
- El **objeto** es la instancia en ejecución (contenedor).

```
Imagen (receta)      →  docker run  →  Contenedor (plato en la mesa)
Clase (plantilla)    →  instanciar  →  Objeto (en memoria)
```

---

## 4. ◈ Instalación de Docker

Las siguientes instrucciones son para referencia. Las celdas de bash de abajo asumen que Docker ya está instalado.

### Windows — WSL 2 + Docker Desktop (Recomendado)

En Windows, el flujo recomendado es **Docker Desktop + WSL 2**. Docker Desktop instala GUI, CLI, Docker Engine y Compose; WSL 2 aporta un kernel Linux integrado sin gestionar una VM manual. Docker recomienda WSL 2 para la mayoría de usuarios en Windows y permite ejecutar comandos docker desde terminales de Windows o desde una distro WSL integrada.

#### 1. Instalación en Windows

##### 1.1 Activar WSL 2

Abrir PowerShell como administrador:

```powershell
wsl --install
```

Reiniciar Windows.

Verificar:

```powershell
wsl -l -v
```

Actualizar WSL:

```powershell
wsl --update
```

Microsoft indica que `wsl --install` instala lo necesario para usar WSL y que las nuevas distribuciones instaladas con ese comando quedan en WSL 2 por defecto.

##### 1.2 Instalar Docker Desktop
Instalar [Docker Desktop for Windows](https://docs.docker.com/desktop/install/windows-install/) desde la documentación oficial.  
Abrir Docker Desktop.  
Ir a:
Settings → General → **Use WSL 2 based engine**  
Ir a:
Settings → Resources → **WSL Integration**  
Activar integración con Ubuntu u otra distro WSL.

Docker documenta que, con WSL 2 activado, los comandos docker quedan disponibles desde terminales Windows usando el engine WSL 2.

> **Nota:** En empresas grandes, el uso comercial de Docker Desktop puede requerir suscripción paga según criterios de Docker.

#### 2. Verificación inicial

En PowerShell o Windows Terminal:

```powershell
docker version
docker info
docker run hello-world
```

Si `hello-world` funciona, Docker ya puede descargar imágenes y ejecutar contenedores.

---

### macOS

Descargar Docker Desktop desde [la documentación oficial](https://www.docker.com/products/docker-desktop):
- Mac Intel → versión Intel
- Mac M1/M2/M3 → versión Apple Silicon

### Linux (Ubuntu/Debian)

```bash
curl -fsSL https://get.docker.com | sh
sudo usermod -aG docker $USER
# Cerrar sesión y volver a entrar
```

---

## 5. ⚡ Comandos básicos — el ciclo de vida de un contenedor

Los comandos de Docker se ejecutan en la **terminal del sistema** (no en Python). Las celdas siguientes usan el prefijo `!` para correrlos desde Jupyter/Colab.

En producción real los correrán directamente en la terminal.

In [ ]:
# Verificamos que Docker esté instalado y vemos su versión:
!docker --version

In [ ]:
# El contenedor más famoso del mundo.
# Docker va a:
# 1. Buscar la imagen 'hello-world' localmente
# 2. No encontrarla → descargarla de Docker Hub
# 3. Crear un contenedor desde esa imagen
# 4. Ejecutarlo → imprime un mensaje y termina
!docker run hello-world

In [ ]:
# Listamos los contenedores que están corriendo AHORA MISMO:
# (hello-world ya terminó, así que la lista puede estar vacía)
!docker ps

In [ ]:
# Con -a vemos TODOS los contenedores, incluso los que ya terminaron:
!docker ps -a

In [ ]:
# Listamos las imágenes descargadas en nuestra máquina:
# hello-world debería aparecer acá
!docker images

### Descargar una imagen sin correrla

`docker pull` descarga la imagen y la guarda localmente. La próxima vez que hagamos `docker run` con esa imagen, no necesita descargarla de nuevo.

In [ ]:
# Descargamos la imagen oficial de Python 3.12 en su versión 'slim'
# (slim = imagen reducida, sin herramientas extras que no necesitamos)
!docker pull python:3.12-slim

In [ ]:
# Corremos un comando de Python directamente dentro del contenedor:
# --rm  → eliminar el contenedor automáticamente cuando termine
# python:3.12-slim → la imagen
# python -c '...'  → el comando a ejecutar dentro del contenedor
!docker run --rm python:3.12-slim python -c "import sys; print('Python', sys.version)"

### ✎ Para pensar

- ¿Qué versión de Python reportó el contenedor? ¿Es la misma que tienen instalada en su máquina?
- Si corrieran el mismo contenedor en otra computadora, ¿qué versión esperarían ver?
- ¿Qué pasa con la imagen que descargaron si la usan con `--rm`? ¿Se borra también?

---

## 6. ◈ Entorno local vs. entorno aislado — la diferencia concreta

```
┌──────────────────────────────────────────────────────────────┐
│                     SU COMPUTADORA                           │
│                                                              │
│  ┌────────────────┐   ┌──────────────────────────────────┐  │
│  │  Entorno local │   │           Docker                 │  │
│  │                │   │  ┌─────────────────────────────┐ │  │
│  │  Python 3.12   │   │  │        Contenedor           │ │  │
│  │  uv / venv     │   │  │                             │ │  │
│  │  deps propias  │   │  │  Python 3.12 (fijo)         │ │  │
│  │                │   │  │  google-genai 1.x           │ │  │
│  │  (varía entre  │   │  │  gradio 5.x                 │ │  │
│  │   máquinas)    │   │  │                             │ │  │
│  └────────────────┘   │  └─────────────────────────────┘ │  │
│                        └──────────────────────────────────┘  │
└──────────────────────────────────────────────────────────────┘
```

**Entorno local:** ideal para desarrollo activo. Rápido de modificar, pero puede diferir entre máquinas.

**Contenedor:** ideal para distribución y despliegue. Idéntico en cualquier máquina con Docker.

> **✦ No son rivales:** desarrollamos en entorno local, distribuimos con contenedores.

---

## 7. ✦ Variables de entorno en Docker

### Por qué no hardcodeamos las API keys

Hardcodear significa escribir el valor directamente en el código:

```python
# ✗ Nunca así:
clave = "AIzaSyABCDEFGHIJKLMNOPQRSTUVWXYZ123456"
```

El problema: si ese código sube a GitHub (aunque sea en un repositorio privado), la clave queda expuesta. Una vez filtrada, hay que revocarla y generar una nueva.

La solución: las claves viven en **variables de entorno**, fuera del código.

### El par `.env` / `.env.example`

```
.env.example   ← plantilla pública (se sube al repositorio)
.env           ← valores reales (NUNCA se sube al repositorio)
```

El `.gitignore` excluye `.env` automáticamente.

### Pasar variables al contenedor

In [ ]:
# Mostramos el contenido del archivo de ejemplo:
# (Este archivo SÍ va al repositorio — no tiene valores reales)
!cat .env.example

In [ ]:
import os
from dotenv import load_dotenv

# load_dotenv() lee el archivo .env y carga cada línea
# como variable de entorno del proceso actual:
load_dotenv()

# Ahora podemos leer la variable con os.getenv():
clave_api = os.getenv("GEMINI_API_KEY")

# Verificamos que la clave esté cargada (sin mostrar el valor real):
if clave_api:
    print("✓ GEMINI_API_KEY cargada correctamente.")
    print(f"  Primeros caracteres: {clave_api[:8]}...")
else:
    print("✗ No se encontró GEMINI_API_KEY. Revisá el archivo .env")

### Cómo pasan las variables al contenedor

Cuando corremos el contenedor, le pasamos el archivo `.env` con `--env-file`:

```bash
# Docker lee el .env y pasa cada variable al entorno del contenedor:
docker run --env-file .env -p 7860:7860 mi-app
```

Dentro del contenedor, `os.getenv("GEMINI_API_KEY")` encuentra la clave como si hubiera hecho `load_dotenv()`, porque Docker la inyectó directamente en el entorno del proceso.

---

## 8. ◈ Dockerfile — escribir la receta de nuestra imagen

Un `Dockerfile` es un archivo de texto, sin extensión, que Docker lee para construir una imagen. Cada línea es una instrucción. Cada instrucción genera una **capa**.

### Por qué el orden importa — capas y caché

Docker guarda en caché cada capa. Si la capa no cambió desde la última vez, no la reconstruye. Si cambió, reconstruye esa capa y todas las que vienen después.

```
FROM python:3.12-slim      ← capa 1: rara vez cambia
COPY requirements.txt .    ← capa 2: cambia cuando agregamos dependencias
RUN pip install ...        ← capa 3: solo se recalcula si cambió la 2
COPY app.py .              ← capa 4: cambia con el código
CMD ["python", "app.py"]   ← capa 5
```

Por eso copiamos `requirements.txt` **antes** que el código: así `pip install` no se vuelve a ejecutar cada vez que modificamos `app.py`.

In [ ]:
# Creamos la carpeta del proyecto de ejemplo:
import os

carpeta_proyecto = "mi-app-gradio"

# Creamos la carpeta si no existe:
os.makedirs(carpeta_proyecto, exist_ok=True)

print(f"✓ Carpeta '{carpeta_proyecto}' lista.")

In [ ]:
# Escribimos el requirements.txt del proyecto:
contenido_requirements = """google-genai>=1.0.0
python-dotenv>=1.0.0
gradio>=5.0.0
"""

ruta_requirements = f"{carpeta_proyecto}/requirements.txt"

with open(ruta_requirements, "w") as archivo:
    archivo.write(contenido_requirements)

print(f"✓ {ruta_requirements} escrito.")
print()
print("Contenido:")
print(contenido_requirements)

In [ ]:
# Escribimos la app de Gradio que va a correr dentro del contenedor:
contenido_app = '''import os
import gradio as gr
from dotenv import load_dotenv
from google import genai

# Cargamos las variables de entorno del archivo .env:
load_dotenv()

# Leemos la clave de API desde el entorno:
clave_api = os.getenv("GEMINI_API_KEY")

# Inicializamos el cliente de Gemini:
cliente = genai.Client(api_key=clave_api)


def responder(mensaje, historial):
    """Recibe el mensaje del usuario y devuelve la respuesta del modelo."""

    # Convertimos el historial del formato de Gradio al formato de la API:
    mensajes_api = []
    for turno in historial:
        # Cada turno es una tupla (mensaje_usuario, mensaje_modelo):
        mensajes_api.append({"role": "user", "parts": [turno[0]]})
        mensajes_api.append({"role": "model", "parts": [turno[1]]})

    # Agregamos el mensaje actual del usuario al final:
    mensajes_api.append({"role": "user", "parts": [mensaje]})

    # Llamamos a la API con el historial completo:
    respuesta = cliente.models.generate_content(
        model="gemini-2.0-flash",
        contents=mensajes_api
    )

    return respuesta.text


# Construimos la interfaz de chat con Gradio:
interfaz = gr.ChatInterface(
    fn=responder,
    title="✦ Asistente Gemini — Clase 3 UTN",
    description="App de chat construida con Gradio y Gemini API, corriendo en Docker.",
)

# server_name="0.0.0.0" es clave:
# hace que Gradio escuche en todas las interfaces de red.
# Sin esto, el contenedor no es accesible desde afuera.
interfaz.launch(server_name="0.0.0.0", server_port=7860)
'''

ruta_app = f"{carpeta_proyecto}/app.py"

with open(ruta_app, "w") as archivo:
    archivo.write(contenido_app)

print(f"✓ {ruta_app} escrito.")

In [ ]:
# Escribimos el Dockerfile:
contenido_dockerfile = """# Imagen base: Python 3.12 en su versión slim (reducida)
FROM python:3.12-slim

# Directorio de trabajo dentro del contenedor:
WORKDIR /app

# Copiamos requirements.txt PRIMERO para aprovechar el caché de capas:
COPY requirements.txt .

# Instalamos las dependencias:
# --no-cache-dir evita guardar archivos de caché de pip (reduce el tamaño de la imagen)
RUN pip install --no-cache-dir -r requirements.txt

# Copiamos el código de la app:
COPY app.py .

# Documentamos el puerto que expone la app:
EXPOSE 7860

# Comando que se ejecuta cuando el contenedor arranca:
CMD ["python", "app.py"]
"""

ruta_dockerfile = f"{carpeta_proyecto}/Dockerfile"

with open(ruta_dockerfile, "w") as archivo:
    archivo.write(contenido_dockerfile)

print(f"✓ {ruta_dockerfile} escrito.")
print()
print("Contenido del Dockerfile:")
print(contenido_dockerfile)

In [ ]:
# Verificamos la estructura del proyecto creado:
import os

archivos_creados = os.listdir(carpeta_proyecto)

print(f"Estructura de '{carpeta_proyecto}':")
for archivo in sorted(archivos_creados):
    print(f"  {archivo}")

---

## 9. ✦ Construir y correr la imagen

Con el proyecto listo, los pasos para correrlo en Docker son tres:

```bash
# 1. Entramos a la carpeta del proyecto:
cd mi-app-gradio

# 2. Construimos la imagen:
# -t le da un nombre a la imagen
# .  indica que el Dockerfile está en la carpeta actual
docker build -t app-gemini-gradio .

# 3. Corremos el contenedor:
# --env-file ../.env  pasa las variables de entorno desde el .env
# -p 7860:7860        mapea el puerto del contenedor al de la máquina
docker run --env-file ../.env -p 7860:7860 app-gemini-gradio
```

Después del tercer comando, abrimos el navegador en: **http://localhost:7860**

La app de chat está corriendo **dentro del contenedor**, no en el sistema operativo.

In [ ]:
# Construimos la imagen (esto puede tardar 1-2 minutos la primera vez):
!docker build -t app-gemini-gradio ./mi-app-gradio

In [ ]:
# Verificamos que la imagen se creó correctamente:
!docker images app-gemini-gradio

Para correr el contenedor, ejecuten en su terminal (no en Jupyter):

```bash
docker run --env-file .env -p 7860:7860 app-gemini-gradio
```

Y abran http://localhost:7860 en el navegador.

---

## 🐛 Laboratorio de Rotura

El código de abajo construye una imagen y la corre. *Parece* correcto. Antes de ejecutarlo, predicción: ¿van a poder acceder a la app en http://localhost:7860?

```python
# app_rota.py
import gradio as gr

def saludar(nombre):
    return f"Hola, {nombre}!"

interfaz = gr.Interface(fn=saludar, inputs="text", outputs="text")

# Corremos la app:
interfaz.launch()  # ← algo está mal acá
```

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir gradio
COPY app_rota.py .
EXPOSE 7860
CMD ["python", "app_rota.py"]
```

Antes de avanzar, respondan:

- ¿Pudieron acceder a la app?
- ¿El contenedor reportó algún error, o corrió sin problemas?
- Si el contenedor corrió bien pero la app no fue accesible desde afuera, ¿qué puede ser diferente entre este caso y el anterior?

**No miren la resolución todavía. Primero la hipótesis.**

### Resolución del Laboratorio de Rotura

El problema está en `interfaz.launch()` sin argumentos.

Por defecto, Gradio escucha en `127.0.0.1` (localhost). Eso significa que solo acepta conexiones desde **dentro del mismo proceso** o del mismo sistema.

Cuando Gradio corre dentro de un contenedor, `127.0.0.1` es el localhost **del contenedor**, no de nuestra máquina. El mapeo de puertos de Docker (`-p 7860:7860`) redirecciona el tráfico desde nuestra máquina al contenedor, pero si Gradio no está escuchando en `0.0.0.0` dentro del contenedor, el tráfico llega y nadie lo atiende.

```python
# ✗ Solo accesible desde dentro del contenedor:
interfaz.launch()

# ✓ Accesible desde afuera:
interfaz.launch(server_name="0.0.0.0", server_port=7860)
```

`0.0.0.0` significa: "escuchá en todas las interfaces de red disponibles". Es el ajuste estándar para cualquier servidor que corra dentro de un contenedor.

---

## 10. ◈ HuggingFace Spaces — despliegue en la nube

HuggingFace Spaces permite publicar la misma app Gradio en internet sin administrar un servidor. El flujo básico:

1. Crear cuenta en https://huggingface.co
2. Ir a **Spaces → Create new Space**
3. Elegir **Gradio** como SDK
4. Subir `app.py` y `requirements.txt`
5. En **Settings → Variables and Secrets**, agregar `GEMINI_API_KEY` como secreto

Spaces tarda unos minutos en construir el entorno. Una vez lista, tiene una URL pública que podemos compartir.

### ✦ La conexión con lo que aprendimos hoy

Spaces hace internamente lo mismo que hicimos: crea un contenedor con el código y lo expone. Solo que el servidor lo administra HuggingFace, no nosotros. El concepto es idéntico.

> **✎ Para explorar:** El `Dockerfile` que escribimos hoy es compatible con Spaces si elegimos "Docker" como SDK en vez de "Gradio". Probarlo es el reto de la semana.

---

## 11. ◈ Referencia rápida

| Comando | Qué hace |
|---|---|
| `docker pull imagen` | Descarga una imagen de Docker Hub |
| `docker run --rm imagen` | Corre y elimina el contenedor al terminar |
| `docker run -d -p 7860:7860 imagen` | Corre en segundo plano, expone puerto |
| `docker run --env-file .env imagen` | Pasa variables de entorno desde archivo |
| `docker ps` | Lista contenedores en ejecución |
| `docker ps -a` | Lista todos los contenedores |
| `docker stop nombre` | Detiene un contenedor |
| `docker rm nombre` | Elimina un contenedor detenido |
| `docker images` | Lista las imágenes descargadas |
| `docker build -t nombre .` | Construye una imagen desde el Dockerfile |
| `docker logs -f nombre` | Muestra logs en tiempo real |
| `docker exec -it nombre sh` | Abre terminal dentro del contenedor |

---

## ⛰️ Diario de Navegación

Cerramos mirando hacia adentro, no hacia el código. Respondan en una o dos líneas, para ustedes:

1. ¿Qué concepto de hoy les costó más encajar en la cabeza? ¿Por qué creen que se resistió?
2. Si tuvieran que explicarle este cuaderno a un colega en lo que dura un viaje en ascensor, ¿qué le dirían?

No hay respuesta correcta. Lo que escriban es el mapa de su propio recorrido.

---

*Próximo bloque → `003_gemini` — Python + Gemini API: prompts, funciones, bucles y visión*